# Data Exploration & Pipeline Verification

Accelerated MRI reconstruction promises faster scan times, but learned models can fabricate anatomical details that never existed in the acquired data. These fabrications, called **hallucinations**, are invisible to standard quality metrics like PSNR and SSIM ([Peck et al., 2026](https://arxiv.org/abs/2602.18536)).

This notebook builds the foundation: raw k-space loading, Fourier utilities, undersampling masks, and the mathematical framework (null-space decomposition) that will serve as ground truth for hallucination detection in later notebooks.

**What this notebook covers:**
1. Cartesian MRI forward model and Fourier utilities
2. Undersampling masks following Lustig et al. (2007)
3. Bhadra et al. (2021) null-space decomposition on synthetic and real data
4. U-Net architecture verification on synthetic phantoms
5. Real fastMRI knee singlecoil data exploration and baseline metrics

**Dataset:** [fastMRI](https://fastmri.med.nyu.edu/) knee singlecoil (Zbontar et al., 2018). All processing starts from raw k-space HDF5 files, never from processed images ([Shimron et al., 2022](https://doi.org/10.1073/pnas.2117203119)).

## Setup
This notebook runs on **Google Colab** with data stored on Google Drive.

**Before running**, make sure the fastMRI knee singlecoil validation set is at:
- `singlecoil_val/` → `/content/drive/MyDrive/fastmri/singlecoil_val/`


In [ ]:
# Install dependencies
!pip install fastmri h5py scikit-image pyyaml tqdm -q

import os, yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import h5py
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# --------------- Paths ---------------
FASTMRI_DATA_DIR = os.environ.get("FASTMRI_DATA_DIR", "/content/data/")
os.makedirs(FASTMRI_DATA_DIR, exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("configs", exist_ok=True)

# --------------- Config ---------------
DEFAULT_CONFIG = {
    'data': {
        'dataset': 'fastmri_knee_singlecoil',
        'data_dir': FASTMRI_DATA_DIR,
        'acceleration': 4,
        'center_fraction': 0.08,
        'mask_types': ['random', 'equispaced', 'gaussian', 'poisson_disc'],
        'image_size': [320, 320],
    },
    'model': {
        'architecture': 'unet',
        'channels': [32, 64, 128, 256],
        'dropout_p': 0.05,
    },
    'training': {
        'epochs': 50, 'lr': 1e-3, 'batch_size': 4, 'loss': 'l1', 'seed': 42,
    },
    'detection': {
        'mc_dropout_samples': 20, 'ensemble_size': 5,
        'sfrc_patch_size': 48, 'sfrc_frc_threshold': 0.75,
    },
    'figures': {
        'dpi': 300, 'font_size_min': 12,
        'cmap_magnitude': 'gray', 'cmap_kspace': 'viridis',
        'cmap_error': 'RdBu_r', 'cmap_detection': 'hot',
        'background': 'white',
    },
}

with open('configs/default.yaml', 'w') as f:
    yaml.dump(DEFAULT_CONFIG, f, default_flow_style=False, sort_keys=False)
with open('configs/default.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# --------------- Figure style (Narnhofer et al. conventions) ---------------
mpl.rcParams.update({
    'figure.dpi': cfg['figures']['dpi'],
    'savefig.dpi': cfg['figures']['dpi'],
    'font.size': cfg['figures']['font_size_min'],
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'figure.facecolor': cfg['figures']['background'],
    'savefig.facecolor': cfg['figures']['background'],
    'savefig.bbox': 'tight',
})

torch.manual_seed(cfg['training']['seed'])
np.random.seed(cfg['training']['seed'])

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --------------- Mount Drive & sync data ---------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Clean slate: remove any truncated files from previous failed syncs
!rm -rf /content/data/singlecoil_val/
!mkdir -p /content/data/singlecoil_val/

import subprocess

MAX_RETRIES = 5
for attempt in range(1, MAX_RETRIES + 1):
    result = subprocess.run(
        ['rsync', '-av', '--progress',
         '/content/drive/MyDrive/fastmri/singlecoil_val/',
         '/content/data/singlecoil_val/'],
        capture_output=True, text=True
    )
    n_local = len(list(Path('/content/data/singlecoil_val/').glob('*.h5')))
    print(f"  Attempt {attempt}: {n_local} files synced")

    if result.returncode == 0:
        break

    print(f"  rsync failed (FUSE drop). Remounting Drive...")
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)

n_final = len(list(Path('/content/data/singlecoil_val/').glob('*.h5')))
print(f"\nVal files ready: {n_final}")
if n_final < 199:
    print(f"WARNING: expected 199 volumes, got {n_final}. Re-run this cell or restart runtime.")

## 1. Cartesian MRI Forward Model

In MRI, raw measurements live in **k-space** (the 2D Fourier transform of the image). Accelerated acquisition skips lines in k-space to reduce scan time. The forward model is:

$$\mathbf{y} = \mathbf{M} \cdot \text{FFT}(\mathbf{x})$$

where $\mathbf{M}$ is a binary mask selecting which k-space lines are acquired. The adjoint (zero-filled IFFT) gives an aliased reconstruction. Content in the **null space** of this operator, meaning the unsampled k-space lines, is invisible to measurements and must be inferred by any reconstruction algorithm.

We use `norm='ortho'` throughout so the FFT/IFFT pair is unitary (adjoint = inverse), which simplifies the null-space math.

In [ ]:
def to_kspace(image: torch.Tensor) -> torch.Tensor:
    """Image -> centered k-space (ortho-normalized)."""
    x = image.to(torch.complex64) if not image.is_complex() else image
    return torch.fft.fftshift(
        torch.fft.fft2(torch.fft.ifftshift(x, dim=(-2, -1)), dim=(-2, -1), norm='ortho'),
        dim=(-2, -1)
    )

def from_kspace(kspace: torch.Tensor) -> torch.Tensor:
    """Centered k-space -> image domain (ortho-normalized)."""
    return torch.fft.fftshift(
        torch.fft.ifft2(torch.fft.ifftshift(kspace, dim=(-2, -1)), dim=(-2, -1), norm='ortho'),
        dim=(-2, -1)
    )

def center_crop(image: torch.Tensor, target_shape: tuple) -> torch.Tensor:
    """Center crop to target shape. Needed for fastMRI: (640, 368) -> (320, 320)."""
    h, w = image.shape[-2], image.shape[-1]
    th, tw = target_shape
    start_h, start_w = (h - th) // 2, (w - tw) // 2
    return image[..., start_h:start_h + th, start_w:start_w + tw]

def reconstruct_and_crop(kspace: torch.Tensor, target_shape: tuple = (320, 320)) -> torch.Tensor:
    """k-space -> IFFT -> magnitude -> center crop."""
    return center_crop(torch.abs(from_kspace(kspace)), target_shape)


class CartesianMRIOperator:
    """Forward model A = mask * FFT. Operates at native k-space resolution."""

    def __init__(self, mask: torch.Tensor):
        self.mask = mask.float()

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        return to_kspace(image) * self.mask

    def adjoint(self, kspace: torch.Tensor) -> torch.Tensor:
        return from_kspace(kspace * self.mask)

    def normal(self, image: torch.Tensor) -> torch.Tensor:
        """A^H A: for Cartesian sampling this is diagonal in k-space."""
        return self.adjoint(self.forward(image))

    def null_space_project(self, image: torch.Tensor) -> torch.Tensor:
        """P_null = I - A^H A. Content surviving this is unobservable from measurements."""
        return image.to(torch.complex64) - self.normal(image)

    def __matmul__(self, image):
        return self.forward(image)


def create_mask(shape, acceleration=4, center_fraction=0.08, mask_type='random', seed=None):
    """Create Cartesian undersampling mask.

    Always keeps a dense center region (low-frequency k-space) and samples
    outer lines according to the chosen strategy. Adapts to any k-space width.
    """
    rng = np.random.RandomState(seed) if seed is not None else np.random.RandomState()

    H, W = shape[-2], shape[-1]
    num_center = int(W * center_fraction)
    num_total = max(int(W / acceleration), num_center)
    num_outer = num_total - num_center

    mask = np.zeros(W, dtype=np.float32)
    center_start = (W - num_center) // 2
    mask[center_start:center_start + num_center] = 1.0
    outer_indices = np.where(mask == 0)[0]

    if mask_type == 'random':
        chosen = rng.choice(outer_indices, size=min(num_outer, len(outer_indices)), replace=False)
        mask[chosen] = 1.0

    elif mask_type == 'equispaced':
        if num_outer > 0 and len(outer_indices) > 0:
            step = max(len(outer_indices) // num_outer, 1)
            offset = rng.randint(0, step) if step > 1 else 0
            chosen = outer_indices[offset::step][:num_outer]
            mask[chosen] = 1.0

    elif mask_type == 'gaussian':
        center = W / 2.0
        sigma = W / 6.0
        probs = np.exp(-0.5 * ((outer_indices - center) / sigma) ** 2)
        probs /= probs.sum()
        chosen = rng.choice(outer_indices, size=min(num_outer, len(outer_indices)),
                            replace=False, p=probs)
        mask[chosen] = 1.0

    elif mask_type == 'poisson_disc':
        if num_outer > 0 and len(outer_indices) > 0:
            min_dist = max(len(outer_indices) / (num_outer * 1.5), 1.0)
            selected = []
            candidates = list(outer_indices)
            rng.shuffle(candidates)
            for c in candidates:
                if len(selected) >= num_outer:
                    break
                if all(abs(c - s) >= min_dist for s in selected):
                    selected.append(c)
            if len(selected) < num_outer:
                remaining = [c for c in outer_indices if c not in selected]
                rng.shuffle(remaining)
                selected += remaining[:num_outer - len(selected)]
            mask[np.array(selected)] = 1.0
    else:
        raise ValueError(f"Unknown mask_type: {mask_type}")

    return torch.from_numpy(mask).unsqueeze(0)

### Operator verification

Two sanity checks on the forward model:
- **Adjoint consistency**: $\langle Ax, y \rangle = \langle x, A^Hy \rangle$ (should hold to machine precision with `norm='ortho'`)
- **Null-space check**: anything projected to the null space should be invisible to measurements, i.e. $\|A \cdot P_{\text{null}}(x)\| \approx 0$

In [ ]:
test_mask = create_mask((320, 320), acceleration=4, mask_type='random', seed=42)
A = CartesianMRIOperator(test_mask)

x_test = torch.randn(320, 320, dtype=torch.complex64)
y_test = torch.randn(320, 320, dtype=torch.complex64)
lhs = torch.sum(A.forward(x_test).conj() * y_test)
rhs = torch.sum(x_test.conj() * A.adjoint(y_test))
print(f"Adjoint consistency: |<Ax,y> - <x,A^Hy>| = {abs(lhs - rhs):.2e}")

null_component = A.null_space_project(x_test)
leaked = torch.max(torch.abs(A.forward(null_component))).item()
print(f"Null-space check:    max|A @ P_null(x)|   = {leaked:.2e}")

# Verify mask adapts to real fastMRI k-space width (368 columns)
mask_real = create_mask((640, 368), acceleration=4, mask_type='random', seed=42)
print(f"\nMask for real data: shape {mask_real.shape}, "
      f"{mask_real.sum().item():.0f}/{mask_real.shape[-1]} lines sampled")

for mt in cfg['data']['mask_types']:
    m = create_mask((320, 320), acceleration=4, mask_type=mt, seed=42)
    print(f"  {mt:15s}: {m.sum().item():.0f}/{m.shape[-1]} lines ({m.sum()/m.numel()*100:.1f}%)")

## 2. Synthetic Phantom and FFT Round-Trip

Before touching real data, we verify the full pipeline on a Shepp-Logan phantom: image -> FFT -> IFFT -> image should recover the original to floating-point precision (~1e-7).

In [ ]:
def create_phantom(size=320):
    """Shepp-Logan style phantom for pipeline testing."""
    img = np.zeros((size, size), dtype=np.float32)
    y, x = np.ogrid[-1:1:size*1j, -1:1:size*1j]

    img += 2.0 * ((x/0.69)**2 + (y/0.92)**2 < 1)
    img -= 0.8 * ((x/0.6624)**2 + ((y+0.0184)/0.874)**2 < 1)
    img += 0.2 * (((x-0.22)/0.11)**2 + (y/0.31)**2 < 1)
    img += 0.2 * (((x+0.22)/0.16)**2 + (y/0.41)**2 < 1)
    img += 0.1 * (((x-0.08)/0.046)**2 + ((y+0.605)/0.046)**2 < 1)
    img += 0.1 * (((x+0.08)/0.046)**2 + ((y+0.605)/0.023)**2 < 1)
    img += 0.1 * ((x/0.046)**2 + ((y+0.605)/0.023)**2 < 1)
    return np.clip(img, 0, None)

phantom = create_phantom(cfg['data']['image_size'][0])
phantom_t = torch.from_numpy(phantom).float()

kspace_t = to_kspace(phantom_t)
roundtrip_t = from_kspace(kspace_t)
roundtrip_error = torch.max(torch.abs(roundtrip_t.real - phantom_t)).item()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('(a) Ground Truth Phantom')
axes[0].axis('off')

im_b = axes[1].imshow(torch.log1p(torch.abs(kspace_t)).numpy(), cmap='viridis')
axes[1].set_title('(b) k-space (log magnitude)')
axes[1].axis('off')
plt.colorbar(im_b, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(roundtrip_t.real.numpy(), cmap='gray')
axes[2].set_title('(c) IFFT Round-Trip')
axes[2].axis('off')

fig.suptitle('FFT Convention Verification', fontsize=14)
plt.tight_layout()
plt.savefig('figures/01_fft_roundtrip.png', bbox_inches='tight')
plt.show()

print(f"Phantom: {phantom.shape}, range [{phantom.min():.3f}, {phantom.max():.3f}]")
print(f"Round-trip max error: {roundtrip_error:.2e}")

![Figure 1](Copie_de_final_knee_mri_1 (4)_figs/fig_001.png)


## 3. Undersampling Masks

Four Cartesian undersampling strategies from the compressed sensing literature (Lustig et al., 2007):

| Mask type | Sampling pattern | Aliasing character |
|-----------|------------------|--------------------|
| **Random** | Uniform random outer lines | Incoherent (noise-like) |
| **Equispaced** | Regular spacing | Coherent (ghost replicas) |
| **Gaussian** | Denser near center | Incoherent, slightly better SNR |
| **Poisson disc** | Blue-noise spacing | Incoherent, uniform coverage |

All masks keep a dense center region (8% of k-space width) to preserve low-frequency contrast.

In [ ]:
mask_types = cfg['data']['mask_types']
accelerations = [4, 8]

fig, axes = plt.subplots(len(mask_types), len(accelerations) * 2,
                          figsize=(4 * len(accelerations) * 2, 3 * len(mask_types)))

for row, mt in enumerate(mask_types):
    for col, acc in enumerate(accelerations):
        mask = create_mask(phantom.shape, acceleration=acc, mask_type=mt, seed=42)
        A = CartesianMRIOperator(mask)
        recon = A.adjoint(A.forward(phantom_t))

        ax_mask = axes[row, col * 2]
        ax_mask.imshow(mask.repeat(20, 1).numpy(), cmap='gray', aspect='auto')
        pct = mask.sum() / mask.numel() * 100
        ax_mask.set_title(f'{mt} {acc}x ({pct:.0f}%)')
        ax_mask.set_yticks([])
        if col == 0:
            ax_mask.set_ylabel(mt, fontsize=12, fontweight='bold')

        ax_recon = axes[row, col * 2 + 1]
        ax_recon.imshow(torch.abs(recon).numpy(), cmap='gray')
        ax_recon.set_title(f'{acc}x recon')
        ax_recon.axis('off')

fig.suptitle('Cartesian Undersampling: 4 Mask Types at 4x and 8x', fontsize=14)
plt.tight_layout()
plt.savefig('figures/01_all_masks_demo.png', bbox_inches='tight')
plt.show()

print("Actual acceleration factors:")
for mt in mask_types:
    for acc in [2, 4, 6, 8]:
        m = create_mask(phantom.shape, acceleration=acc, mask_type=mt, seed=42)
        actual = m.numel() / m.sum().item()
        print(f"  {mt:15s} target {acc}x -> actual {actual:.1f}x")

![Figure 2](Copie_de_final_knee_mri_1 (4)_figs/fig_002.png)


## 4. Image Quality Metrics and Error Maps

Standard metrics used in fastMRI benchmarks (Zbontar et al., 2018):
- **PSNR** (dB): measures pixel-level fidelity
- **SSIM**: captures perceptual structural similarity
- **NMSE**: normalized mean squared error

Peck et al. (2026) showed these metrics **cannot reliably detect hallucinations**. A reconstruction can score well on PSNR/SSIM while containing fabricated anatomical details. We compute them here as baselines, but hallucination detection will require different tools (Notebooks 03 through 05).

In [ ]:
def compute_metrics(gt: np.ndarray, recon: np.ndarray) -> dict:
    """PSNR, SSIM, NMSE between ground truth and reconstruction."""
    gt = np.abs(gt).astype(np.float64)
    recon = np.abs(recon).astype(np.float64)
    data_range = gt.max() - gt.min()
    if data_range == 0:
        return {'psnr': float('inf'), 'ssim': 1.0, 'nmse': 0.0}
    return {
        'psnr': peak_signal_noise_ratio(gt, recon, data_range=data_range),
        'ssim': structural_similarity(gt, recon, data_range=data_range),
        'nmse': np.sum((gt - recon) ** 2) / np.sum(gt ** 2),
    }

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, acc in enumerate([2, 4, 6, 8]):
    mask = create_mask(phantom.shape, acceleration=acc, mask_type='random', seed=42)
    A = CartesianMRIOperator(mask)
    recon = A.adjoint(A.forward(phantom_t))
    recon_np = torch.abs(recon).numpy()
    error = recon_np - phantom
    metrics = compute_metrics(phantom, recon_np)

    axes[0, i].imshow(recon_np, cmap='gray')
    axes[0, i].set_title(f'({chr(97+i)}) {acc}x IFFT')
    axes[0, i].axis('off')

    im = axes[1, i].imshow(error, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[1, i].set_title(f'PSNR={metrics["psnr"]:.1f} dB')
    axes[1, i].axis('off')
    plt.colorbar(im, ax=axes[1, i], fraction=0.046, pad=0.04)

fig.suptitle('Zero-Filled IFFT Reconstructions and Error Maps (random mask)', fontsize=14)
plt.tight_layout()
plt.savefig('figures/01_error_maps.png', bbox_inches='tight')
plt.show()

print(f"{'Mask Type':15s} {'Acc':>4s} {'PSNR (dB)':>10s} {'SSIM':>8s} {'NMSE':>10s}")
print("-" * 52)
for mt in cfg['data']['mask_types']:
    for acc in [4, 8]:
        mask = create_mask(phantom.shape, acceleration=acc, mask_type=mt, seed=42)
        A = CartesianMRIOperator(mask)
        recon = A.adjoint(A.forward(phantom_t))
        m = compute_metrics(phantom, torch.abs(recon).numpy())
        print(f"{mt:15s} {acc:4d}x {m['psnr']:10.1f} {m['ssim']:8.3f} {m['nmse']:10.4f}")

![Figure 3](Copie_de_final_knee_mri_1 (4)_figs/fig_003.png)


## 5. Data Loading

The fastMRI singlecoil HDF5 files contain:

| Key | Shape | Description |
|-----|-------|-------------|
| `kspace` | `(num_slices, 640, W)` complex64 | Raw k-space (W varies: 368, 372, ...) |
| `reconstruction_rss` | `(num_slices, 320, 320)` float32 | Root-sum-of-squares reference (from multicoil) |
| `reconstruction_esc` | `(num_slices, 320, 320)` float32 | Emulated singlecoil reference |

**Important:** `reconstruction_rss` was computed from multicoil data that our singlecoil k-space cannot fully represent, creating a hard ceiling of ~26 dB for IFFT vs RSS. The correct singlecoil reference is `reconstruction_esc`, which matches our IFFT at ~155 dB. We use ESC as the training target in Notebook 02.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class FastMRIDataset(Dataset):
    """fastMRI singlecoil dataset loading raw k-space from HDF5.

    Enforces Shimron et al. (2022) data hygiene: only loads complex k-space,
    rejects non-HDF5 files. IFFT + crop + normalization happens inside
    __getitem__ to handle variable k-space widths across volumes.
    """
    def __init__(self, data_dir, acceleration=4, center_fraction=0.08, mask_type='random'):
        self.data_dir = Path(data_dir)
        self.acceleration = acceleration
        self.center_fraction = center_fraction
        self.mask_type = mask_type
        self.examples = []

        if self.data_dir.exists():
            h5_files = sorted(self.data_dir.glob("*.h5"))
            non_h5 = list(self.data_dir.glob("*.dcm")) + list(self.data_dir.glob("*.nii*"))
            if non_h5:
                print(f"WARNING: Found {len(non_h5)} non-HDF5 files. Ignored.")

            for h5_path in h5_files:
                try:
                    with h5py.File(h5_path, "r") as f:
                        if "kspace" not in f or not np.iscomplexobj(f["kspace"][0]):
                            continue
                        num_slices = f["kspace"].shape[0]
                        self.examples += [(h5_path, i) for i in range(num_slices)]
                except Exception:
                    continue

        print(f"FastMRIDataset: {len(self.examples)} slices from "
              f"{len(set(p for p, _ in self.examples))} volumes")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        h5_path, slice_idx = self.examples[idx]
        with h5py.File(h5_path, "r") as f:
            kspace = torch.from_numpy(f["kspace"][slice_idx].copy())
            target = torch.from_numpy(f["reconstruction_rss"][slice_idx].copy()).float()

        mask = create_mask(kspace.shape, acceleration=self.acceleration,
                           center_fraction=self.center_fraction,
                           mask_type=self.mask_type, seed=idx)
        return kspace * mask, mask, target


class SyntheticMRIDataset(Dataset):
    """Random ellipse phantoms. Drop-in replacement for FastMRIDataset."""
    def __init__(self, num_samples=100, image_size=(320, 320),
                 acceleration=4, mask_type='random'):
        self.num_samples = num_samples
        self.image_size = image_size
        self.acceleration = acceleration
        self.mask_type = mask_type

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        rng = np.random.RandomState(idx)
        h, w = self.image_size
        img = np.zeros((h, w), dtype=np.float32)
        y, x = np.ogrid[-1:1:h*1j, -1:1:w*1j]
        img += 1.0 * ((x/0.9)**2 + (y/0.7)**2 < 1)
        for _ in range(rng.randint(3, 8)):
            cx, cy = rng.uniform(-0.5, 0.5, 2)
            rx, ry = rng.uniform(0.05, 0.25, 2)
            img += rng.uniform(0.3, 0.8) * (((x - cx)/rx)**2 + ((y - cy)/ry)**2 < 1)

        target = torch.from_numpy(img).float()
        kspace = to_kspace(target)
        mask = create_mask(self.image_size, acceleration=self.acceleration,
                           mask_type=self.mask_type, seed=idx)
        return kspace * mask, mask, target


# Quick validation
synth_dataset = SyntheticMRIDataset(num_samples=50, image_size=(320, 320), acceleration=4)
ks, mk, tgt = synth_dataset[0]
print(f"Synthetic: kspace {ks.shape}, mask {mk.shape}, target {tgt.shape}")

val_dir = '/content/data/singlecoil_val/'
if Path(val_dir).exists():
    real_dataset = FastMRIDataset(val_dir, acceleration=4, mask_type='random')
    ks_r, mk_r, tgt_r = real_dataset[len(real_dataset)//2]
    print(f"Real:      kspace {ks_r.shape}, mask {mk_r.shape}, target {tgt_r.shape}")

## 6. Null-Space Decomposition (Bhadra et al., 2021)

The mathematical definition of a hallucination: given forward operator $A = M \cdot \text{FFT}$, any reconstruction $\hat{x}$ can be decomposed as

$$\hat{x} - x^* = \underbrace{A^H(AA^H)^{-1}A(\hat{x} - x^*)}_{\text{measurement error}} + \underbrace{(I - A^HA)(\hat{x} - x^*)}_{\text{null-space error (hallucination)}}$$

For zero-filled IFFT, null-space energy should be **exactly 100%**: the IFFT only contains measured frequencies, so all reconstruction error comes from the missing (null-space) lines. For a U-Net, the null-space portion represents content the network **fabricated** beyond what measurements support.

We validate on synthetic data first (with an injected hallucination blob), then on real fastMRI data.

In [ ]:
def null_space_decomposition(reconstruction, ground_truth, mask):
    """Bhadra et al. (2021) decomposition of reconstruction error.

    Returns energy fractions and error maps for measurement-space
    and null-space (hallucination) components.
    """
    A = CartesianMRIOperator(mask)
    recon_c = reconstruction.to(torch.complex64)
    gt_c = ground_truth.to(torch.complex64)

    total_error = recon_c - gt_c
    meas_error = A.normal(total_error)
    null_error = total_error - meas_error

    # Verification: two paths to null-space should agree
    null_via_projector = A.null_space_project(total_error)
    verification_diff = torch.max(torch.abs(null_error - null_via_projector)).item()

    total_energy = torch.sum(torch.abs(total_error)**2).item()
    null_energy = torch.sum(torch.abs(null_error)**2).item()
    meas_energy = torch.sum(torch.abs(meas_error)**2).item()

    return {
        'total_error': total_error,
        'measurement_error': meas_error,
        'null_space_error': null_error,
        'null_space_energy': null_energy / (total_energy + 1e-10),
        'measurement_energy': meas_energy / (total_energy + 1e-10),
        'verification_diff': verification_diff,
        'energy_additivity_error': abs(total_energy - null_energy - meas_energy) / (total_energy + 1e-10),
    }


def null_space_decomposition_cropped(kspace_recon, kspace_full, mask, crop_shape=(320, 320)):
    """Null-space decomposition with center-cropped outputs for visualization."""
    recon_full = from_kspace(kspace_recon)
    gt_full = from_kspace(kspace_full)
    result = null_space_decomposition(recon_full, gt_full, mask)

    for key in ['total_error', 'measurement_error', 'null_space_error']:
        result[key + '_cropped'] = center_crop(result[key], crop_shape)
    result['recon_cropped'] = center_crop(torch.abs(recon_full), crop_shape)
    result['gt_cropped'] = center_crop(torch.abs(gt_full), crop_shape)
    return result

### Synthetic validation: injected hallucination

We add a Gaussian blob to the IFFT reconstruction and verify the decomposition correctly identifies it as null-space content.

In [ ]:
mask_4x = create_mask(phantom.shape, acceleration=4, mask_type='random', seed=42)
A = CartesianMRIOperator(mask_4x)
ifft_recon = A.adjoint(A.forward(phantom_t))

# Inject a blob that simulates hallucinated content
yy, xx = torch.meshgrid(torch.arange(320), torch.arange(320), indexing='ij')
blob = 0.5 * torch.exp(-((yy - 160)**2 + (xx - 240)**2) / (2 * 15**2))
fake_recon = ifft_recon + blob.to(torch.complex64)

result = null_space_decomposition(fake_recon, phantom_t, mask_4x)

print(f"Projector consistency:  {result['verification_diff']:.2e}")
print(f"Energy additivity:     {result['energy_additivity_error']:.2e}")
print(f"Measurement energy:    {result['measurement_energy']*100:.1f}%")
print(f"Null-space energy:     {result['null_space_energy']*100:.1f}%")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(phantom, cmap='gray')
axes[0, 0].set_title('(a) Ground Truth'); axes[0, 0].axis('off')

axes[0, 1].imshow(torch.abs(fake_recon).numpy(), cmap='gray')
axes[0, 1].set_title('(b) Reconstruction + injected blob'); axes[0, 1].axis('off')

im = axes[0, 2].imshow(torch.abs(result['total_error']).numpy(), cmap='hot', vmin=0, vmax=0.5)
axes[0, 2].set_title('(c) |Total Error|'); axes[0, 2].axis('off')
plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

im = axes[1, 0].imshow(torch.abs(result['measurement_error']).numpy(), cmap='hot', vmin=0, vmax=0.5)
axes[1, 0].set_title(f'(d) Measurement Error ({result["measurement_energy"]*100:.1f}%)')
axes[1, 0].axis('off')
plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

im = axes[1, 1].imshow(torch.abs(result['null_space_error']).numpy(), cmap='hot', vmin=0, vmax=0.5)
axes[1, 1].set_title(f'(e) Hallucination Map ({result["null_space_energy"]*100:.1f}%)')
axes[1, 1].axis('off')
plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

im = axes[1, 2].imshow(torch.abs(result['null_space_error']).numpy(), cmap='hot', vmin=0, vmax=0.5)
axes[1, 2].set_title('(f) Null-Space + Injection Site'); axes[1, 2].axis('off')
plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)
circle = plt.Circle((240, 160), 30, fill=False, color='cyan', linewidth=2, linestyle='--')
axes[1, 2].add_patch(circle)
axes[1, 2].annotate('injected\nhallucination', xy=(240, 160), xytext=(270, 100),
                     color='cyan', fontsize=11, fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color='cyan', lw=2))

fig.suptitle('Null-Space Decomposition (Synthetic Phantom)', fontsize=14)
plt.tight_layout()
plt.savefig('figures/01_null_space_decomposition.png', bbox_inches='tight')
plt.show()

![Figure 4](Copie_de_final_knee_mri_1 (4)_figs/fig_004.png)


### Real data: zero-filled IFFT decomposition

On real fastMRI data, the IFFT reconstruction yields **100% null-space energy**. This is mathematically correct: zero-filled IFFT = $A^H(Ax)$, which by definition only contains measured frequencies. All error lives in the unsampled k-space lines.

The interesting case comes with learned reconstructions (Notebook 02+), where the network adds content in both measurement space and null space. The null-space additions are hallucinations.

In [ ]:
val_dir = '/content/data/singlecoil_val/'
h5_files = sorted(Path(val_dir).glob('*.h5'))

with h5py.File(h5_files[0], 'r') as f:
    mid = f['kspace'].shape[0] // 2
    kspace_full_real = torch.from_numpy(f['kspace'][mid].copy())
    target_real = torch.from_numpy(f['reconstruction_rss'][mid].copy()).float()

mask_real = create_mask(kspace_full_real.shape, acceleration=4, mask_type='random', seed=42)
kspace_under_real = kspace_full_real * mask_real

result_real = null_space_decomposition_cropped(
    kspace_under_real, kspace_full_real, mask_real, crop_shape=(320, 320)
)

print(f"Real fastMRI knee (4x IFFT):")
print(f"  Measurement energy:  {result_real['measurement_energy']*100:.1f}%")
print(f"  Null-space energy:   {result_real['null_space_energy']*100:.1f}%")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
vmax = torch.abs(result_real['total_error_cropped']).quantile(0.99).item()

axes[0, 0].imshow(target_real.numpy(), cmap='gray')
axes[0, 0].set_title('(a) Ground Truth (RSS)'); axes[0, 0].axis('off')

axes[0, 1].imshow(result_real['recon_cropped'].numpy(), cmap='gray')
axes[0, 1].set_title('(b) 4x Zero-filled IFFT'); axes[0, 1].axis('off')

im = axes[0, 2].imshow(torch.abs(result_real['total_error_cropped']).numpy(), cmap='hot', vmin=0, vmax=vmax)
axes[0, 2].set_title('(c) |Total Error|'); axes[0, 2].axis('off')
plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

im = axes[1, 0].imshow(torch.abs(result_real['measurement_error_cropped']).numpy(), cmap='hot', vmin=0, vmax=vmax)
axes[1, 0].set_title(f'(d) Measurement Error ({result_real["measurement_energy"]*100:.1f}%)')
axes[1, 0].axis('off')
plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

im = axes[1, 1].imshow(torch.abs(result_real['null_space_error_cropped']).numpy(), cmap='hot', vmin=0, vmax=vmax)
axes[1, 1].set_title(f'(e) Hallucination Map ({result_real["null_space_energy"]*100:.1f}%)')
axes[1, 1].axis('off')
plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

axes[1, 2].imshow(target_real.numpy(), cmap='gray')
axes[1, 2].imshow(torch.abs(result_real['null_space_error_cropped']).numpy(),
                   cmap='hot', alpha=0.5, vmin=0, vmax=vmax)
axes[1, 2].set_title('(f) Null-Space Overlaid on Anatomy'); axes[1, 2].axis('off')

fig.suptitle('Null-Space Decomposition on Real fastMRI Knee (4x IFFT)', fontsize=14)
plt.tight_layout()
plt.savefig('figures/01_null_space_real_data.png', bbox_inches='tight')
plt.show()

![Figure 5](Copie_de_final_knee_mri_1 (4)_figs/fig_005.png)


## 7. U-Net Architecture

Standard encoder-decoder with skip connections (Ronneberger et al., 2015). We use a lightweight configuration with channels `[32, 64, 128, 256]` and `Dropout2d(p=0.05)` baked in. The dropout is a no-op in eval mode but will be activated for MC Dropout inference in Notebook 05.

The network operates on 320x320 center-cropped magnitude images normalized to [0,1]. A quick training loop on synthetic phantoms verifies the architecture handles the full pipeline (k-space -> IFFT -> crop -> normalize -> U-Net -> loss).

Real training on fastMRI data happens in Notebook 02.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    """U-Net for MRI reconstruction, operates on 320x320 magnitude images."""
    def __init__(self, channels=(32, 64, 128, 256), dropout_p=0.05):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        self.dropout = nn.Dropout2d(p=dropout_p)

        in_ch = 1
        for ch in channels:
            self.encoders.append(ConvBlock(in_ch, ch))
            self.pools.append(nn.MaxPool2d(2))
            in_ch = ch

        self.bottleneck = ConvBlock(channels[-1], channels[-1] * 2)

        for ch in reversed(channels):
            self.upconvs.append(nn.ConvTranspose2d(ch * 2, ch, 2, stride=2))
            self.decoders.append(ConvBlock(ch * 2, ch))

        self.final = nn.Conv2d(channels[0], 1, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x)
            skips.append(x)
            x = pool(x)
            x = self.dropout(x)

        x = self.bottleneck(x)

        for upconv, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x)
            if x.shape != skip.shape:
                x = nn.functional.pad(x, [0, skip.shape[3] - x.shape[3],
                                          0, skip.shape[2] - x.shape[2]])
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            x = self.dropout(x)

        return self.final(x)


def ifft_to_input(kspace_under, crop_shape=(320, 320)):
    """Undersampled k-space -> normalized magnitude image for U-Net input."""
    mag = torch.abs(from_kspace(kspace_under))
    if mag.shape[-2] != crop_shape[0] or mag.shape[-1] != crop_shape[1]:
        mag = center_crop(mag, crop_shape)
    mag = mag.unsqueeze(1)
    B = mag.shape[0]
    flat = mag.reshape(B, -1)
    mins = flat.min(dim=1, keepdim=True).values.reshape(B, 1, 1, 1)
    maxs = flat.max(dim=1, keepdim=True).values.reshape(B, 1, 1, 1)
    return (mag - mins) / (maxs - mins + 1e-8)

def normalize_target(target):
    """Normalize ground truth to [0,1] per sample."""
    if target.dim() == 3:
        target = target.unsqueeze(1)
    B = target.shape[0]
    flat = target.reshape(B, -1)
    mins = flat.min(dim=1, keepdim=True).values.reshape(B, 1, 1, 1)
    maxs = flat.max(dim=1, keepdim=True).values.reshape(B, 1, 1, 1)
    return (target - mins) / (maxs - mins + 1e-8)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(channels=tuple(cfg['model']['channels']),
             dropout_p=cfg['model']['dropout_p']).to(device)

param_count = sum(p.numel() for p in model.parameters())
print(f"U-Net: {param_count:,} parameters")

# Quick training on synthetic data (pipeline verification only)
optimizer = optim.Adam(model.parameters(), lr=cfg['training']['lr'])
criterion = nn.L1Loss()
synth_train = SyntheticMRIDataset(num_samples=200, image_size=(320, 320), acceleration=4)
train_loader = DataLoader(synth_train, batch_size=4, shuffle=True, num_workers=0)

for epoch in range(5):
    model.train()
    epoch_loss = 0.0
    for kspace_under, mask, target in tqdm(train_loader, desc=f"Epoch {epoch+1}/5"):
        input_img = ifft_to_input(kspace_under).to(device)
        target_img = normalize_target(target).to(device)
        pred = model(input_img)
        loss = criterion(pred, target_img)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"  Epoch {epoch+1}: L1 = {epoch_loss / len(train_loader):.4f}")

# Visualize one sample
model.eval()
with torch.no_grad():
    ks, mk, tgt = synth_train[0]
    inp = ifft_to_input(ks.unsqueeze(0)).to(device)
    pred = model(inp).cpu().squeeze()
    tgt_norm = normalize_target(tgt.unsqueeze(0)).squeeze()

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].imshow(inp.cpu().squeeze().numpy(), cmap='gray')
axes[0].set_title('(a) Zero-filled IFFT'); axes[0].axis('off')
axes[1].imshow(pred.numpy(), cmap='gray')
axes[1].set_title('(b) U-Net Output'); axes[1].axis('off')
axes[2].imshow(tgt_norm.numpy(), cmap='gray')
axes[2].set_title('(c) Ground Truth'); axes[2].axis('off')
im = axes[3].imshow(pred.numpy() - tgt_norm.numpy(), cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[3].set_title('(d) Error'); axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
fig.suptitle('U-Net on Synthetic Data (5 epochs, pipeline check)', fontsize=14)
plt.tight_layout()
plt.savefig('figures/01_unet_verification.png', bbox_inches='tight')
plt.show()

m = compute_metrics(tgt_norm.numpy(), pred.numpy())
print(f"Synthetic: PSNR={m['psnr']:.1f} dB, SSIM={m['ssim']:.3f}")

![Figure 6](Copie_de_final_knee_mri_1 (4)_figs/fig_006.png)


## 8. Real fastMRI Knee Data

First look at a real singlecoil volume. Key things to verify:
- `reconstruction_esc` matches our IFFT of the full k-space (confirming pipeline correctness)
- `reconstruction_rss` has a ~26 dB ceiling from the singlecoil/multicoil mismatch
- Undersampled IFFT metrics provide a baseline for the U-Net to beat

In [ ]:
val_dir = '/content/data/singlecoil_val/'
h5_files = sorted(Path(val_dir).glob('*.h5'))
print(f"Validation set: {len(h5_files)} volumes")

with h5py.File(h5_files[0], 'r') as f:
    print(f"\nFile: {h5_files[0].name}")
    print(f"  kspace:             {f['kspace'].shape} {f['kspace'].dtype}")
    print(f"  reconstruction_rss: {f['reconstruction_rss'].shape}")
    if 'reconstruction_esc' in f:
        print(f"  reconstruction_esc: {f['reconstruction_esc'].shape}")

with h5py.File(h5_files[0], 'r') as f:
    mid = f['kspace'].shape[0] // 2
    kspace_real = torch.from_numpy(f['kspace'][mid].copy())
    target_rss = torch.from_numpy(f['reconstruction_rss'][mid].copy()).float()
    target_esc = None
    if 'reconstruction_esc' in f:
        target_esc = torch.from_numpy(f['reconstruction_esc'][mid].copy()).float()

full_recon = reconstruct_and_crop(kspace_real, (320, 320))
mask_4x = create_mask(kspace_real.shape, acceleration=4, mask_type='random', seed=42)
recon_4x = reconstruct_and_crop(kspace_real * mask_4x, (320, 320))

m_full = compute_metrics(target_rss.numpy(), full_recon.numpy())
m_4x = compute_metrics(target_rss.numpy(), recon_4x.numpy())

print(f"\nFull IFFT vs RSS: PSNR={m_full['psnr']:.1f} dB  (singlecoil ceiling)")
print(f"4x IFFT vs RSS:  PSNR={m_4x['psnr']:.1f} dB")
if target_esc is not None:
    m_esc = compute_metrics(target_esc.numpy(), full_recon.numpy())
    print(f"Full IFFT vs ESC: PSNR={m_esc['psnr']:.1f} dB  (pipeline correctness check)")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(target_rss.numpy(), cmap='gray')
axes[0].set_title('(a) Ground Truth (RSS)'); axes[0].axis('off')

im = axes[1].imshow(torch.log1p(torch.abs(kspace_real)).numpy(), cmap='viridis')
axes[1].set_title('(b) Fully Sampled k-space'); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(recon_4x.numpy(), cmap='gray')
axes[2].set_title('(c) 4x Zero-filled IFFT'); axes[2].axis('off')

error = recon_4x.numpy() - target_rss.numpy()
vmax = np.percentile(np.abs(error), 99)
im = axes[3].imshow(error, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[3].set_title(f'(d) Error (PSNR={m_4x["psnr"]:.1f} dB)'); axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

plt.suptitle('Real fastMRI Knee Singlecoil', fontsize=14)
plt.tight_layout()
plt.savefig('figures/01_real_data_first_look.png', bbox_inches='tight')
plt.show()

![Figure 7](Copie_de_final_knee_mri_1 (4)_figs/fig_007.png)


### Baseline statistics across the validation set

We evaluate IFFT reconstruction at 4x on the middle 50% of slices (avoiding uninformative edge slices near the top/bottom of the knee volume) across 10 volumes. These numbers provide the baseline that the U-Net must beat in Notebook 02.

In [ ]:
psnrs, ssims = [], []
n_volumes = min(10, len(h5_files))

for h5_path in h5_files[:n_volumes]:
    with h5py.File(h5_path, 'r') as f:
        n_slices = f['kspace'].shape[0]
        for s in range(n_slices // 4, 3 * n_slices // 4):
            ks = torch.from_numpy(f['kspace'][s].copy())
            tgt = torch.from_numpy(f['reconstruction_rss'][s].copy()).float()
            mk = create_mask(ks.shape, acceleration=4, mask_type='random', seed=s)
            rec = reconstruct_and_crop(ks * mk, (320, 320))
            m = compute_metrics(tgt.numpy(), rec.numpy())
            psnrs.append(m['psnr'])
            ssims.append(m['ssim'])

print(f"4x IFFT baseline ({len(psnrs)} slices from {n_volumes} volumes):")
print(f"  PSNR: {np.mean(psnrs):.1f} ± {np.std(psnrs):.1f} dB")
print(f"  SSIM: {np.mean(ssims):.3f} ± {np.std(ssims):.3f}")

## Summary

| Result | Value | Status |
|--------|-------|--------|
| FFT round-trip error | ~1e-7 | ✓ Pipeline correct |
| Adjoint consistency | ~3e-5 | ✓ Ortho norm works |
| Null-space projector | ~7e-7 | ✓ Correct |
| Synthetic null-space (injected blob) | ~95% null | ✓ Blob detected |
| Real IFFT null-space energy | 100% | ✓ Mathematically expected |
| Full IFFT vs ESC | ~155 dB | ✓ Singlecoil pipeline correct |
| Full IFFT vs RSS | ~26 dB | Multicoil ceiling (expected) |
| 4x IFFT vs RSS | ~24 ± 3 dB | Baseline for U-Net |

Everything checks out. The data pipeline is sound, the Fourier conventions are correct, and the null-space decomposition framework is validated.

**Next:** Notebook 02 trains a U-Net on real fastMRI data to produce reconstructions that contain hallucinations, then we characterize and detect them in Notebooks 03 through 05.

In [ ]:
import json

notebook_path = '/content/drive/MyDrive/collab_transfer/Copie de final_knee_mri_1.ipynb'

with open(notebook_path, 'r') as f:
    nb = json.load(f)

if 'metadata' in nb and 'widgets' in nb['metadata']:
    del nb['metadata']['widgets']
    print("Widgets stripped.")
else:
    print("No widget metadata found.")

with open(notebook_path, 'w') as f:
    json.dump(nb, f, indent=1)

print("Done. Download this file and upload to GitHub.")

In [ ]:
import json

# Find the file first
import glob
files = glob.glob('/content/drive/MyDrive/collab_transfer/*.ipynb')
print("Found:", files)

if files:
    notebook_path = files[0]

    with open(notebook_path, 'r') as f:
        raw = f.read()

    nb = json.loads(raw)

    # 1. Strip top-level widget metadata
    if 'metadata' in nb:
        nb['metadata'].pop('widgets', None)

    # 2. Strip widget state from every cell
    for cell in nb.get('cells', []):
        cell.get('metadata', {}).pop('widgets', None)

        # 3. Remove tqdm widget outputs from cell outputs
        if 'outputs' in cell:
            cleaned = []
            for out in cell['outputs']:
                # Skip widget display outputs
                if out.get('output_type') == 'display_data':
                    if 'application/vnd.jupyter.widget-view+json' in out.get('data', {}):
                        continue
                cleaned.append(out)
            cell['outputs'] = cleaned

    # Save cleaned version
    clean_path = notebook_path.replace('.ipynb', '_clean.ipynb')
    with open(clean_path, 'w') as f:
        json.dump(nb, f, indent=1)

    print(f"\nCleaned notebook saved to:\n{clean_path}")
    print("Download this _clean file and upload to GitHub.")